In [2]:
import torch
import triton
import triton.language as tl

In [3]:
@triton.jit
def add_vectors(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE):
    pid = tl.program_id(axis=0) # Get the current block index in the x-axis (must do for every axis that we are dealing with, 1D only in this case)
    block_start = pid * BLOCK_SIZE # Get the set of threads that are involved in this current run
    offsets = tl.arange(block_start, block_start + BLOCK_SIZE) # Get the threads involved in this operation
    x = x_ptr + offsets # Get all the values in the corresponding range for the threads
    y = y_ptr + offsets
    # Masking, important for the last block, if n_elements is not divisible by block_size, then there will be more threads active than elements. So this prevents the extra threads from running
    # E.g. if block_size = 8, grid_size = 4 blocks (so 32 threads), n_elements = 28
    # Last block mask (indices 24-31): True, True, True, True, False, False, False, False -> So no operations done on indices 28-31
    mask = offsets < n_elements 
    tl.load(x, mask = mask)
    tl.load(y, mask = mask) 
    output = x + y
    tl.store(out_ptr + offsets, output, mask = mask)